# ChurnIQ: Telecom Customer Churn & Revenue Risk Analyzer

## Phase 2: Data Quality Assessment

### Objective

Evaluate the quality of the dataset before any cleaning or preprocessing.

This notebook focuses on:

- Duplicate detection
- Missing value assessment
- Hidden missing values
- Data consistency checks
- Potential data leakage identification
- Data quality reporting

No data cleaning will be performed in this notebook.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
## 2. Load Dataset

In [3]:
train_df = pd.read_csv("../data/raw/cell2celltrain.csv")
holdout_df = pd.read_csv("../data/raw/cell2cellholdout.csv")

## 3. Duplicate Row Assessment

### Objective

Identify completely duplicated records that may bias model training and evaluation.

In [5]:
train_df.duplicated().sum()

np.int64(0)

## 4. Duplicate CustomerID Assessment

### Objective

Verify whether the same customer appears multiple times in the dataset.

In [7]:
train_df["CustomerID"].duplicated().sum()

np.int64(0)

## 5. Missing Value Percentage Analysis

### Objective

Measure the percentage of missing values in each feature.

In [8]:
missing_pct = (
    train_df.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_pct[missing_pct > 0]

AgeHH2                   1.780712
AgeHH1                   1.780712
PercChangeMinutes        0.718945
PercChangeRevenues       0.718945
MonthlyRevenue           0.305601
MonthlyMinutes           0.305601
TotalRecurringCharge     0.305601
DirectorAssistedCalls    0.305601
OverageMinutes           0.305601
RoamingCalls             0.305601
ServiceArea              0.047015
CurrentEquipmentDays     0.001959
Handsets                 0.001959
HandsetModels            0.001959
dtype: float64

### Observation

No duplicate CustomerIDs were found.

Each row represents a unique customer.

## 6. Hidden Missing Value Assessment

### Objective

Identify categorical values that may represent missing or unknown information but are not stored as null values.

In [10]:
potential_missing_values = [
    "Unknown",
    "Other",
    "Known"
]

for value in potential_missing_values:
    print(f"\nChecking value: {value}")

    for col in train_df.select_dtypes(include="object").columns:

        count = (train_df[col] == value).sum()

        if count > 0:
            print(f"{col}: {count}")


Checking value: Unknown
Homeownership: 17060
HandsetPrice: 28982
MaritalStatus: 19700

Checking value: Other
PrizmCode: 24655
Occupation: 37637

Checking value: Known
Homeownership: 33987


### Hidden Missing Value Findings

Several categorical variables contain values such as "Unknown" and "Other".

Key observations:

- MaritalStatus contains a large proportion of Unknown values.
- Homeownership contains both Known and Unknown categories.
- Occupation and PrizmCode contain a dominant Other category.
- HandsetPrice contains a large number of Unknown values and requires further investigation.

These values may require special treatment during the data cleaning phase.

## 7. Potential Data Leakage Assessment

### Objective

Identify variables that may contain information unavailable at prediction time and could artificially inflate model performance.

In [11]:
leakage_candidates = [
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

train_df[leakage_candidates].head()

,RetentionCalls,RetentionOffersAccepted,MadeCallToRetentionTeam
0,1,0,Yes
1,0,0,No
2,0,0,No
3,0,0,No
4,0,0,No


In [12]:
train_df[leakage_candidates].describe()

,RetentionCalls,RetentionOffersAccepted
count,51047.000000,51047.000000
mean,0.037201,0.018277
std,0.206483,0.142458
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,4.000000,3.000000


In [13]:
for col in leakage_candidates:
    print(f"\n{col}")
    print(train_df[col].value_counts(dropna=False))


RetentionCalls
RetentionCalls
0    49302
1     1609
2      120
3       14
4        2
Name: count, dtype: int64

RetentionOffersAccepted
RetentionOffersAccepted
0    50166
1      837
2       36
3        8
Name: count, dtype: int64

MadeCallToRetentionTeam
MadeCallToRetentionTeam
No     49302
Yes     1745
Name: count, dtype: int64


In [14]:
train_df["HandsetPrice"].dtype

dtype('O')

### Observation

HandsetPrice is currently stored as an object datatype rather than a numeric datatype.

A large number of "Unknown" values are present, which likely prevents automatic numeric interpretation.

This issue will be addressed during the data cleaning phase.

## 8. Leakage Risk Investigation

### Objective

Assess whether retention-related variables are strongly associated with churn and may introduce leakage.

In [15]:
pd.crosstab(
    train_df["MadeCallToRetentionTeam"],
    train_df["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
MadeCallToRetentionTeam,,
No,71.75571,28.24429
Yes,54.95702,45.04298


In [16]:
pd.crosstab(
    train_df["RetentionCalls"],
    train_df["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
RetentionCalls,,
0,71.755710,28.244290
1,55.003108,44.996892
2,55.833333,44.166667
3,50.000000,50.000000
4,0.000000,100.000000


In [17]:
pd.crosstab(
    train_df["RetentionOffersAccepted"],
    train_df["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
RetentionOffersAccepted,,
0,71.396962,28.603038
1,59.020311,40.979689
2,55.555556,44.444444
3,62.500000,37.500000


## 9. Data Quality Summary

In [18]:
train_df.select_dtypes(include="object").columns.tolist()

['Churn',
 'ServiceArea',
 'ChildrenInHH',
 'HandsetRefurbished',
 'HandsetWebCapable',
 'TruckOwner',
 'RVOwner',
 'Homeownership',
 'BuysViaMailOrder',
 'RespondsToMailOffers',
 'OptOutMailings',
 'NonUSTravel',
 'OwnsComputer',
 'HasCreditCard',
 'NewCellphoneUser',
 'NotNewCellphoneUser',
 'OwnsMotorcycle',
 'HandsetPrice',
 'MadeCallToRetentionTeam',
 'CreditRating',
 'PrizmCode',
 'Occupation',
 'MaritalStatus']

## Data Quality Summary

### Key Findings

- No duplicate records were found.
- No duplicate CustomerIDs were found.
- Missing values are present but generally low (<2%).
- Several categorical variables contain values such as "Unknown" and "Other".
- HandsetPrice is stored as an object datatype due to the presence of non-numeric values.
- Retention-related variables show a strong association with churn and have been flagged for further leakage assessment during modeling.

### Overall Assessment

The dataset demonstrates good overall quality with low missingness and no duplication issues. A small number of fields require special treatment during data cleaning and feature engineering.